In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# Ô 2 — Nạp code, dữ liệu, 300 URL độc (giống tuần 6)
for f in ['models/thresholds_D1.json', 'models/thresholds_D2.json', 'models/RF_F1D2_D1.joblib', 'src/d2.py']:
    assert os.path.exists(f'{ROOT}/{f}'), f'Thiếu {f}: cả hai phải xong tuần 6 trước.'
import common; importlib.reload(common)
from common import *
import json, joblib, random, time
import matplotlib.pyplot as plt
A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
B = pd.read_parquet(f'{ROOT}/data/processed/B.parquet')
tr, va, te = (A.split == 'train').values, (A.split == 'val').values, (A.split == 'test').values
import attacks, e3, defenses, d2
for m in (attacks, e3, defenses, d2): importlib.reload(m)
from attacks import *
from e3 import *
from defenses import d1_normalize
from d2 import make_F1D2
benign_domains = set(A[A.label == 0]['reg_domain'])

# Cùng 300 URL độc như tuần 6 (cùng seed) -> so sánh được
N = 300
rng0 = random.Random(0)
MAL = {'A-test': rng0.sample([to_attack_form(u) for u in A[te & (A.label == 1).values]['url_norm']], N),
       'B':      rng0.sample([to_attack_form(u) for u in B[B.label == 1]['url_norm']], N)}
ALL_OPS = FREE_OPS + PAID_OPS
print({k: len(v) for k, v in MAL.items()})

In [ ]:
# Ô 3 — Ghi file src/e4.py: nạp 5 cấu hình phòng thủ (dùng chung cho Nhựt và Trí)
E4_CODE = r"""
import json, os, joblib
from common import make_F1, evaluate
from d2 import make_F1D2
from defenses import d1_normalize
from e3 import make_score_fn

FEAT = {'F1': make_F1, 'F1D2': make_F1D2}
#           tên cấu hình   file model        file ngưỡng           key ngưỡng       feature  có D1?
CONFIGS = [('none',      'RF_F1',          'thresholds.json',    'RF_F1',          'F1',   False),
           ('D1',        'RF_F1_D1',       'thresholds_D1.json', 'RF_F1_D1',       'F1',   True),
           ('D2',        'RF_F1D2',        'thresholds_D2.json', 'RF_F1D2',        'F1D2', False),
           ('D1+D2',     'RF_F1D2_D1',     'thresholds_D2.json', 'RF_F1D2_D1',     'F1D2', True),
           ('D1+D2+D3',  'RF_F1D2_D1_D3',  'thresholds_D3.json', 'RF_F1D2_D1_D3',  'F1D2', True)]

def load_configs(root):
    out = {}
    for name, mfile, tfile, tkey, fs, use_d1 in CONFIGS:
        mp, tp = f'{root}/models/{mfile}.joblib', f'{root}/models/{tfile}'
        if not (os.path.exists(mp) and os.path.exists(tp)):
            print('  (chưa có cấu hình', name, '- bỏ qua)'); continue
        m = joblib.load(mp); pre = d1_normalize if use_d1 else None
        out[name] = {'thr': json.load(open(tp))[tkey], 'score': make_score_fn(m, FEAT[fs], pre)}
    return out

def clean_eval(cfg, df):
    # Chấm điểm qua đúng đường ống của cấu hình (D1 nếu có) trên URL sạch
    s = cfg['score'](['http://' + u for u in df['url_norm']])
    return evaluate(df['label'], s, cfg['thr'])
"""
with open(f'{ROOT}/src/e4.py', 'w', encoding='utf-8') as f:
    f.write(E4_CODE)
import e4; importlib.reload(e4)
from e4 import load_configs, clean_eval
print('Đã ghi', f'{ROOT}/src/e4.py')

In [ ]:
# Ô 4 — D3: huấn luyện đối kháng. Thêm URL độc đã biến đổi ngẫu nhiên vào TRAIN (chỉ train, chỉ nguồn A)
# Giữ lại T6 (typo): KHÔNG dùng khi train, để kiểm tra phòng thủ có tổng quát hóa sang biến đổi chưa thấy không
from sklearn.base import clone
HELD_OUT = 'T6'

def augment_train(train_df, frac=0.5, seed=0):
    rng = random.Random(seed)
    ops = [o for o in TRANSFORMS if o != HELD_OUT]
    mal = train_df[train_df.label == 1].sample(frac=frac, random_state=seed)
    adv = [random_attack(to_attack_form(u), rng.choice([1, 2, 3]), ops, rng, benign_domains)[0] for u in mal['url_norm']]
    aug = pd.DataFrame({'url_norm': [normalize_url(a) for a in adv], 'label': 1})
    return pd.concat([train_df[['url_norm', 'label']], aug], ignore_index=True), len(aug)

def d1_frame(df):
    u = [normalize_url(d1_normalize('http://' + x)) for x in df['url_norm']]
    out = pd.DataFrame({'url_norm': u, 'label': df['label'].values})
    parts = out['url_norm'].map(split_parts)
    out['host'] = parts.str[0]; out['rest'] = parts.str[1]
    return out

t0 = time.time()
train_aug, n_aug = augment_train(A[tr], frac=0.5, seed=0)
Xtr, Xva = make_F1D2(d1_frame(train_aug)), make_F1D2(d1_frame(A[va]))
rf_d3 = clone(joblib.load(f'{ROOT}/models/RF_F1D2_D1.joblib')).fit(Xtr, train_aug.label)
thr_d3 = threshold_at_fpr(A.label[va], get_scores(rf_d3, Xva))
joblib.dump(rf_d3, f'{ROOT}/models/RF_F1D2_D1_D3.joblib')
json.dump({'RF_F1D2_D1_D3': thr_d3}, open(f'{ROOT}/models/thresholds_D3.json', 'w'), indent=2)
print(f'Thêm {n_aug:,} URL độc đã biến đổi vào train | ngưỡng D3 = {thr_d3:.4f} | {time.time() - t0:.0f}s')

In [ ]:
# Ô 5 — E4 (seed 0): 5 cấu hình × (sạch A, sạch B, tấn công tham lam, tấn công ngẫu nhiên, chỉ T6)
CFG = load_configs(ROOT)
rows = []
for name, cfg in CFG.items():
    t0 = time.time()
    cA, cB = clean_eval(cfg, A[te]), clean_eval(cfg, B)
    log_result(ROOT, 'E4_clean', 'RF', 'A-test', cA, defense=name)
    log_result(ROOT, 'E4_clean', 'RF', 'B', cB, defense=name)
    g, _ = attack_success_rate(MAL['A-test'], cfg['score'], cfg['thr'], 3, ALL_OPS, 'greedy', benign_domains, seed=0)
    r, _ = attack_success_rate(MAL['A-test'], cfg['score'], cfg['thr'], 3, ALL_OPS, 'random', benign_domains, seed=0)
    h, _ = attack_success_rate(MAL['A-test'], cfg['score'], cfg['thr'], 1, [HELD_OUT], 'greedy', benign_domains, seed=0)
    for mode, m, ops in [('greedy', g, 'all'), ('random', r, 'all'), ('greedy', h, HELD_OUT)]:
        log_result(ROOT, 'E4', 'RF', 'A-test', m, defense=name, seed=0, k=3 if ops == 'all' else 1, ops=ops, mode=mode)
    rows.append({'cấu hình': name, 'F1 sạch A': cA['f1'], 'FPR A': cA['fpr'], 'F1 B': cB['f1'], 'FPR B': cB['fpr'],
                 'gap F1': cA['f1'] - cB['f1'], 'recall sạch': g['recall_clean'],
                 'recall tham lam': g['recall_attack'], 'ASR tham lam': g['asr'], 'ASR ngẫu nhiên': r['asr'],
                 'ASR chỉ T6': h['asr'], 'số truy vấn TB': g['mean_queries']})
    print(f'{name} xong ({time.time() - t0:.0f}s)')
E4 = pd.DataFrame(rows).set_index('cấu hình')
E4.to_csv(f'{ROOT}/results/E4_seed0.csv')
print(E4.round(3).T.to_string())

In [ ]:
# Ô 6 — 20 URL độc VẪN né được cấu hình mạnh nhất (D1+D2+D3): biến đổi nào, vì sao lọt?
best = 'D1+D2+D3' if 'D1+D2+D3' in CFG else list(CFG)[-1]
_, res = attack_success_rate(MAL['A-test'], CFG[best]['score'], CFG[best]['thr'], 3, ALL_OPS, 'greedy', benign_domains, seed=0)
ev = res[res.evaded].head(20)
lines = [f"{'+'.join(r.ops) or '(không cần sửa)'} | {r.queries} truy vấn | {defang(r.adv_url)}" for r in ev.itertuples()]
open(f'{ROOT}/results/E4_evaded_20.txt', 'w', encoding='utf-8').write('\n'.join(lines))
print(f'{best}: {res.evaded.sum()} / {len(res)} URL né được. 20 mẫu đầu:\n')
print('\n'.join(lines))
print('\nThống kê biến đổi được dùng:', pd.Series([o for ops in ev.ops for o in ops]).value_counts().to_dict())